In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/rent_payments.csv"
TARGET_SCHEMA = "property_catalog.property_management"

RAW_TABLE = f"{TARGET_SCHEMA}.raw_rent_payments"
CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_rent_payments"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_rent_payments"

In [0]:
def extract_rent_payments(source_path):
    rent_payments_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(source_path)
    )

    print("Rent payments extracted successfully!")

    # Log the number of source records
    print(f"Number of rows: {rent_payments_df.count()}")

    return rent_payments_df

rent_payments_df = extract_rent_payments(SOURCE_PATH)

# Save raw rent payments to a managed Delta table
(
    rent_payments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(RAW_TABLE)
)

display(rent_payments_df)

In [0]:
def transform_rent_payments(rent_payments_df):
    # Standardise rent payments columns and data types
    transformed_df = (
        rent_payments_df
        .withColumn("Id", F.expr("try_cast(trim(Id) AS BIGINT)"))
        .withColumn("Amount", F.expr("try_cast(trim(Amount) AS DECIMAL(12,2))"))
        .withColumn("PaymentDate", F.expr("try_cast(trim(PaymentDate) AS DATE)"))
        .withColumn("Status", F.initcap(F.trim(F.col("Status"))))
    )

    print("Rent payments transformation completed!")
    print(f"Total records: {transformed_df.count()}")

    display(transformed_df)

    return transformed_df

# Read raw rent payments Delta table
rent_payments_df = spark.table(RAW_TABLE)

transformed_rent_payments_df = transform_rent_payments(rent_payments_df)

In [0]:
def validate_rent_payments(transformed_rent_payments_df):
    validated_df = (
        transformed_rent_payments_df
        # Flag duplicate property IDs
        .withColumn("duplicate_payment_id", F.count("*").over(Window.partitionBy("Id")) > 1)

        # Flag records with missing required values
        .withColumn(
            "has_missing_values",
            F.col("Id").isNull()
            | F.col("Amount").isNull()
            | F.col("PaymentDate").isNull()
            | F.col("Status").isNull()
            | (F.trim(F.col("Status")) == "")
        )

        # Flag invalid payment amounts
        .withColumn("invalid_amount", F.col("Amount").isNull() | (F.col("Amount") <= 0))

        # Flag invalid payment dates
        .withColumn("invalid_payment_date", F.col("PaymentDate").isNull())
    )

    print("Rent payments validation completed!")
    
    return validated_df

validated_rent_payments_df = validate_rent_payments(transformed_rent_payments_df)

display(validated_rent_payments_df)

In [0]:
def separate_rent_payments(validated_rent_payments_df):
    rejection_condition = (
        F.col("duplicate_payment_id")
        | F.col("has_missing_values")
        | F.col("invalid_amount")
        | F.col("invalid_payment_date")
    )

    # Filter rejected rent payment records and add detailed reasons for each validation failure
    rejected_rent_payments_df = (
        validated_rent_payments_df
        .filter(rejection_condition)
        .withColumn(
            "rejection_reason",
            F.concat_ws(
                ". ",
                F.when(F.col("duplicate_payment_id"), F.lit("Duplicate payment ID")),
                F.when(F.col("has_missing_values"), F.lit("Missing required values")),
                F.when(F.col("invalid_amount"), F.lit("Payment amount must be greater than zero")),
                F.when(F.col("invalid_payment_date"), F.lit("Invalid payment date"))
            )
        )
    )

    # Filter clean rent payment records
    clean_rent_payments_df = validated_rent_payments_df.filter(~rejection_condition)

    return clean_rent_payments_df, rejected_rent_payments_df

clean_rent_payments_df, rejected_rent_payments_df = separate_rent_payments(validated_rent_payments_df)

print("Clean rent payments:")
display(clean_rent_payments_df)

print("Rejected rent payments:")
display(rejected_rent_payments_df)

In [0]:
def summarise_rent_payments(rent_payments_df, clean_rent_payments_df, rejected_rent_payments_df):
    total_count = rent_payments_df.count()
    clean_count = clean_rent_payments_df.count()
    rejected_count = rejected_rent_payments_df.count()

    # Calculate the clean record success rate
    success_rate = (
        clean_count / total_count * 100
        if total_count > 0
        else 0
    )

    print("=" * 45)
    print("RENT PAYMENTS ETL SUMMARY")
    print("=" * 45)
    print(f"Source: {SOURCE_PATH}")
    print(f"Total records: {total_count}")
    print(f"Clean records: {clean_count}")
    print(f"Rejected records: {rejected_count}")
    print(f"Success rate: {success_rate:.2f}%")
    print("=" * 45)

    return {
        "total": total_count,
        "clean": clean_count,
        "rejected": rejected_count,
        "success_rate": success_rate
    }

rent_payments_summary = summarise_rent_payments(rent_payments_df, clean_rent_payments_df, rejected_rent_payments_df)


In [0]:
def load_rent_payments(clean_rent_payments_df, rejected_rent_payments_df):
    # Remove validation flags from the clean output
    clean_rent_payments_output_df = clean_rent_payments_df.drop(
        "duplicate_payment_id",
        "has_missing_values",
        "invalid_amount",
        "invalid_payment_date"
    )
    
    # Save clean rent payments to a managed Delta table
    (
        clean_rent_payments_output_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CLEAN_TABLE)
    )

    # Save rejected rent payments to a managed Delta table
    (
        rejected_rent_payments_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REJECTED_TABLE)
    )

    print(f"Clean rent payments records saved to: {CLEAN_TABLE}")
    print(f"Rejected rent payments records saved to: {REJECTED_TABLE}")

load_rent_payments(clean_rent_payments_df, rejected_rent_payments_df)